# 08b — Nelson Mandela Hybrid GraphRAG: Query

1. Semantic search via lexical-graph (Wikipedia content)
2. Structured queries via document-graph (events, organizations)
3. Hybrid correlation — combine both


In [1]:
# Setup
import json, re
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory, VectorStoreFactory
from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from document_graph.query import DocumentGraphQueryEngine

GRAPH_STORE = 'neptune-db://obs-app-dev-graph.cluster-czupj1ab2tc6.us-east-1.neptune.amazonaws.com:8182'
VECTOR_STORE = 'aoss://https://1lci0mi6xy7hpex8fr0b.us-east-1.aoss.amazonaws.com'
TENANT = 'mandela'

graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE)
vector_store = VectorStoreFactory.for_vector_store(VECTOR_STORE)
gs = graph_store.__enter__()
print('Connected')


Connected


## 1. Semantic search (lexical-graph)


In [2]:
query_engine = LexicalGraphQueryEngine.for_traversal_based_search(graph_store, vector_store)

queries = [
    'Who was Nelson Mandela?',
    'What role did Mandela play in ending apartheid?',
    'What is the African National Congress?',
]

for q in queries:
    print(f'\nQuery: {q}')
    results = query_engine.retrieve(q)
    if isinstance(results, list) and results:
        print(f'  Results: {len(results)}')
        text = getattr(results[0].node, 'text', str(results[0]))[:200]
        print(f'  Top: {text}...')
    else:
        print(f'  {str(results)[:150]}')



Query: Who was Nelson Mandela?
  Results: 5
  Top: {
  "source": "Nelson Mandela (wikipedia)",
  "topic": "Nelson Mandela Life, Career, and Legacy",
  "statements": [
    "[Nelson Mandela joined the African National Congress in 1943.]",
    "[Nelson R...

Query: What role did Mandela play in ending apartheid?
  Results: 5
  Top: {
  "source": "African National Congress (wikipedia)",
  "topic": "African National Congress (ANC) Overview",
  "statements": [
    "[The African National Congress has been the ruling party of post-ap...

Query: What is the African National Congress?
  Results: 5
  Top: {
  "source": "African National Congress (wikipedia)",
  "topic": "African National Congress (ANC) Overview",
  "statements": [
    "[The African National Congress has been the ruling party of post-ap...


## 2. Structured queries (document-graph)


In [4]:
# Query events
event_label = f"__Event__{TENANT}__"
events = gs.execute_query(f"MATCH (n:`{event_label}`) RETURN properties(n) as props LIMIT 10")
print(f"Events in document-graph: {len(events)}")
for e in events[:5]:
    props = e.get("props", {})
    date = props.get("date", "")
    desc = props.get("description", "?")
    print(f"  [{date}] {desc}")

# Query organizations
org_label = f"__Organization__{TENANT}__"
orgs = gs.execute_query(f"MATCH (n:`{org_label}`) RETURN properties(n) as props LIMIT 10")
print(f"Organizations: {len(orgs)}")
for o in orgs[:5]:
    props = o.get("props", {})
    name = props.get("name", props.get("title", "?"))
    print(f"  {name}")


Events in document-graph: 9
  [2013-12-05] Passed away at age 95.
  [1918-07-18] Nelson Rolihlahla Mandela was born.
  [1944-01-01] Joined African National Congress (ANC) and co-founded the ANC Youth League.
  [1952-06-26] Led Defiance Campaign against unjust apartheid laws.
  [1962-08-05] Arrested and later sentenced to five years for leaving the country illegally and inciting strike.
Organizations: 5
  The Evolution of Apartheid Policies, 1948–1994
  Resistance Networks and the ANC
  International Sanctions and South Africa
  Truth and Reconciliation: Outcomes and Limits
  Media Representation of Mandela


## 3. Hybrid correlation


In [ ]:
# Semantic query + correlate entities to document-graph
q = 'What key events shaped Mandela\'s life?'
results = query_engine.retrieve(q)
print(f'Query: {q}')
print(f'Lexical results: {len(results)}\n')

# Extract entities and look them up in document-graph
for i, r in enumerate(results[:3], 1):
    meta = getattr(r.node, 'metadata', {})
    ec = meta.get('entity_contexts', {}).get('contexts', [])
    text = getattr(r.node, 'text', str(r))[:150]
    print(f'Result {i}: {text}...')

    for ctx in ec[:5]:
        for ent in ctx.get('entities', []):
            entity = ent.get('entity', {})
            value = entity.get('value', '')
            classification = entity.get('classification', '')
            if not value:
                continue
            search_term = value.split(',')[0].strip()
            found = None
            if classification in ('Event', 'Organization', 'Person'):
                for label in [f'__Event__{TENANT}__', f'__Organization__{TENANT}__']:
                    found = gs.execute_query(f"MATCH (n:`{label}`) WHERE n.name CONTAINS '{search_term}' OR n.title CONTAINS '{search_term}' RETURN properties(n) as props LIMIT 1")
                    if found:
                        break
            if found:
                props = found[0].get('props', {})
                name = props.get('name', props.get('title', value))
                print(f'    [{classification}] "{value}" -> Neptune: {name}')
    print()

print('Hybrid query complete — lexical search + document-graph correlation')


## Summary

| Layer | Source | Contains |
|-------|--------|----------|
| Lexical-graph | Wikipedia | Semantic chunks, topics, entities from Mandela bio |
| Document-graph | Structured CSV/JSON | Events, organizations as typed nodes |
| Hybrid query | Both | Semantic search finds context, entity correlation finds structured data |
